# 01. Setup & Baseline (Text Cell)

**Phase 1 → Phase 2 진입**: HuggingFace datasets 로드 → DSC 베이스라인 (clean) → 모델 5종 베이스라인 metric 수집

DSC v5 framework — text × classification (ADR-016) + text × regression (ADR-017) 사전등록.

분류 튜닝 dataset 3종: `fancyzhx/ag_news` / `stanfordnlp/imdb` / `SetFit/20_newsgroups`
회귀 튜닝 dataset 3종: `Yelp/yelp_review_full` (50K) / `mteb/amazon_reviews_multi` 'en' (200K) / `SetFit/sst5`

---


## 0. 환경 설정

Colab T4 GPU 가정. 학교 카드 결제 활성화 + HuggingFace 토큰 발급(필요 시) 완료 전제.


In [ ]:
# requirements-text.txt 패키지 설치 (Colab 환경)
!pip install -q transformers>=4.30 datasets>=2.10 xgboost>=1.7 sentence-transformers

import sys, os, json
ROOT = '/content/drive/MyDrive/capstone/dsc'  # Colab Google Drive 마운트 경로 (조정)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset

from dsc_framework.text_cell import compute_dsc_text
from dsc_framework.text_cell_regression import compute_dsc_text_regression

print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'n/a')


## 1. 튜닝 dataset 로드

ADR-016 §3-1 / ADR-017 §3-1 freeze.


In [ ]:
# 분류 트랙
ag_news = load_dataset('fancyzhx/ag_news')
imdb    = load_dataset('stanfordnlp/imdb')
news20  = load_dataset('SetFit/20_newsgroups')

# 회귀 트랙 (Yelp/Amazon은 sample_cap 적용)
yelp    = load_dataset('Yelp/yelp_review_full')
amazon  = load_dataset('mteb/amazon_reviews_multi', 'en')
sst5    = load_dataset('SetFit/sst5')

print('sizes:',
    {'ag_news': len(ag_news['train']), 'imdb': len(imdb['train']),
     '20news': len(news20['train']), 'yelp': len(yelp['train']),
     'amazon': len(amazon['train']), 'sst5': len(sst5['train'])})


## 2. 샘플링 (ADR-017 §4 freeze)

Yelp / Amazon은 stratified random_state=42, train 50K (Yelp) / 200K (Amazon) / test 5K.


In [ ]:
def stratified_sample(ds, label_key, n_per_split, seed=42):
    rng = np.random.RandomState(seed)
    df = ds.to_pandas()
    parts = [g.sample(min(len(g), n_per_split // df[label_key].nunique()), random_state=seed)
             for _, g in df.groupby(label_key)]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

yelp_train_s  = stratified_sample(yelp['train'],   'label',  50000)
yelp_test_s   = stratified_sample(yelp['test'],    'label',   5000)
amazon_train_s= stratified_sample(amazon['train'], 'label', 200000)
amazon_test_s = stratified_sample(amazon['test'],  'label',   5000)
print('Yelp:', len(yelp_train_s), len(yelp_test_s), '| Amazon:', len(amazon_train_s), len(amazon_test_s))


## 3. DSC 베이스라인 (clean DSC, default 가중치)

ADR-015 원칙: 본 단계의 default DSC는 합격선 출발점 측정. 운영 가중치는 Phase 4 LLM 호출 결과 사용.


In [ ]:
def df_to_text_label(ds, text_key='text', label_key='label'):
    if isinstance(ds, pd.DataFrame):
        return ds[text_key].tolist(), ds[label_key].tolist()
    return ds[text_key], ds[label_key]

# 분류 cell — clean baseline
for name, ds in [('ag_news', ag_news['train']),
                  ('imdb',   imdb['train']),
                  ('20news', news20['train'])]:
    texts, labels = df_to_text_label(ds)
    r = compute_dsc_text(texts[:5000], labels[:5000],
                         use_embeddings=True, sample_cap=1000, random_state=42)
    print(f"{name:8s}  DSC={r['score']}  grade={r['grade']}")

# 회귀 cell — clean baseline (target=label cast to float)
for name, ds in [('yelp_50k',   yelp_train_s),
                  ('amazon_200k', amazon_train_s),
                  ('sst5',       pd.DataFrame(sst5['train']))]:
    texts, lbs = df_to_text_label(ds)
    targets = [float(x) for x in lbs]
    r = compute_dsc_text_regression(texts[:5000], targets[:5000],
                                    use_embeddings=True, sample_cap=1000, random_state=42)
    print(f"{name:12s} DSC={r['score']}  grade={r['grade']}")


## 4. 모델 베이스라인 — clean 학습

5 모델 × 6 dataset baseline accuracy/R² 수집. 학습 시간 절약 위해 각 dataset의 train_sub(예: 5000)로 sanity. Phase 2 정식 학습은 03 노트북에서 풀 train으로 재실행.

다음 셀들은 GPU 환경에서 실행 — 본 노트북 stub은 함수 정의만 둠.


In [ ]:
# Transformer head 분류 학습 (DistilBERT/BERT/RoBERTa 공유)
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
# 함수 정의는 03_training_text.ipynb에 풀 버전 작성


In [ ]:
# TextCNN 분류 (random init embedding + 3-kernel conv)
# class TextCNN(torch.nn.Module): ...  # 03_training_text.ipynb 참조


In [ ]:
# LogReg + TF-IDF baseline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

def logreg_tfidf_accuracy(train_texts, train_labels, test_texts, test_labels,
                          max_features=20000, ngram_range=(1, 2)):
    vec = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range)
    Xtr = vec.fit_transform(train_texts)
    Xte = vec.transform(test_texts)
    clf = LogisticRegression(max_iter=2000, n_jobs=-1, random_state=42).fit(Xtr, train_labels)
    p = clf.predict(Xte)
    return accuracy_score(test_labels, p), f1_score(test_labels, p, average='macro')


---

다음: `02_pollution_and_dsc_text.ipynb` — 7 polluter × 6 level × 6 dataset 스윕.
